# Analyze ONNX for GPT

Analyze one NeuroGolf ONNX file and generate a compact text package that can be pasted into GPT for optimization ideas.

In [1]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

scripts_dir = repo_root / "scripts"
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

from neurogolf_score import score_model_file
from neurogolf_onnx_analysis import (
    compare_score_rows,
    compact_model_markdown,
    gpt_analysis_prompt,
    run_model_on_examples,
    summarize_model,
    summarize_task,
)

DATA_DIR = repo_root / "neurogolf-2026"
WORK_DIR = repo_root / "outputs" / "gpt_onnx_analysis"
WORK_DIR.mkdir(parents=True, exist_ok=True)

TASK_ID = "task003"
ONNX_PATH = repo_root / "solution" / "6406.18" / f"{TASK_ID}.onnx"
CANDIDATE_ONNX_DIR = repo_root / "outputs" / "gpt_workbench" / TASK_ID
CANDIDATE_ONNX_PATHS = sorted(CANDIDATE_ONNX_DIR.glob("*.onnx"))

TASK_ID, ONNX_PATH, CANDIDATE_ONNX_DIR, CANDIDATE_ONNX_PATHS


('task003',
 PosixPath('/Users/kaiikeda/Programming/Kaggle/Neuro Golf/solution/6406.18/task003.onnx'),
 PosixPath('/Users/kaiikeda/Programming/Kaggle/Neuro Golf/outputs/gpt_workbench/task003'),
 [PosixPath('/Users/kaiikeda/Programming/Kaggle/Neuro Golf/outputs/gpt_workbench/task003/task003_best_v79_baseline.onnx'),
  PosixPath('/Users/kaiikeda/Programming/Kaggle/Neuro Golf/outputs/gpt_workbench/task003/task003_rebuild_v100_out_zero_from_ch0_p2_c02.onnx'),
  PosixPath('/Users/kaiikeda/Programming/Kaggle/Neuro Golf/outputs/gpt_workbench/task003/task003_rebuild_v101_out_zero_from_ch0_p2_c20.onnx'),
  PosixPath('/Users/kaiikeda/Programming/Kaggle/Neuro Golf/outputs/gpt_workbench/task003/task003_rebuild_v102_out_zero_from_ch0_p2_c22.onnx'),
  PosixPath('/Users/kaiikeda/Programming/Kaggle/Neuro Golf/outputs/gpt_workbench/task003/task003_rebuild_v103_out_ch0_neg_add_p2_c00.onnx'),
  PosixPath('/Users/kaiikeda/Programming/Kaggle/Neuro Golf/outputs/gpt_workbench/task003/task003_rebuild_v104_out

## Task Summary

In [2]:
task_summary = summarize_task(TASK_ID, DATA_DIR)

print(f"task    : {task_summary['task']}")
print(f"train   : {task_summary['num_train']}")
print(f"test    : {task_summary['num_test']}")
print(f"arc-gen : {task_summary['num_arc_gen']}")
print(f"colors  : {task_summary['color_counts']}")

try:
    import pandas as pd

    display(pd.DataFrame(task_summary["examples"]))
except ImportError:
    task_summary["examples"]

task    : task003
train   : 3
test    : 1
arc-gen : 261
colors  : {0: 5961, 1: 2386, 2: 3578}


,subset,index,input_shape,output_shape,input_cells,output_cells
0,train,0,6x3,9x3,18,27
1,train,1,6x3,9x3,18,27
2,train,2,6x3,9x3,18,27
3,test,0,6x3,9x3,18,27
4,arc-gen,0,6x3,9x3,18,27
...,...,...,...,...,...,...
260,arc-gen,256,6x3,9x3,18,27
261,arc-gen,257,6x3,9x3,18,27
262,arc-gen,258,6x3,9x3,18,27
263,arc-gen,259,6x3,9x3,18,27


## Score

In [3]:
base_score_row = score_model_file(ONNX_PATH, DATA_DIR)
candidate_score_rows = []

for candidate_path in CANDIDATE_ONNX_PATHS:
    row = score_model_file(candidate_path, DATA_DIR)
    row["candidate_file"] = candidate_path.name
    row["candidate_path"] = str(candidate_path)
    row.update(compare_score_rows(base_score_row, row))
    candidate_score_rows.append(row)

# Keep this alias for the GPT prompt generation cells below.
score_row = base_score_row

print("Base/reference ONNX:")
display(base_score_row)

print(f"Candidate ONNX files: {len(candidate_score_rows)}")
try:
    import pandas as pd

    if candidate_score_rows:
        score_cols = [
            "candidate_file", "status", "score", "score_delta", "cost", "cost_delta",
            "memory", "memory_delta", "params", "params_delta", "filesize", "filesize_delta", "error"
        ]
        display(pd.DataFrame(candidate_score_rows)[score_cols].sort_values("score", ascending=False).reset_index(drop=True))
    else:
        print(f"No candidate ONNX files found in {CANDIDATE_ONNX_DIR}")
except ImportError:
    candidate_score_rows


2026-06-17 01:33:14.773 python[55052:18286619] 2026-06-17 01:33:14.773706 [W:onnxruntime:, graph.cc:4885 CleanUnusedInitializersAndNodeArgs] Removing initializer 'safe_name_16'. It is not used by any node and should be removed from the model.
2026-06-17 01:33:14.774 python[55052:18286619] 2026-06-17 01:33:14.774156 [W:onnxruntime:, graph.cc:4885 CleanUnusedInitializersAndNodeArgs] Removing initializer 'safe_name_15'. It is not used by any node and should be removed from the model.
2026-06-17 01:33:14.852 python[55052:18286619] 2026-06-17 01:33:14.852315 [W:onnxruntime:, graph.cc:4885 CleanUnusedInitializersAndNodeArgs] Removing initializer 'safe_name_14'. It is not used by any node and should be removed from the model.
2026-06-17 01:33:14.852 python[55052:18286619] 2026-06-17 01:33:14.852398 [W:onnxruntime:, graph.cc:4885 CleanUnusedInitializersAndNodeArgs] Removing initializer 'safe_name_13'. It is not used by any node and should be removed from the model.
2026-06-17 01:33:14.886 pyth

Base/reference ONNX:


2026-06-17 01:33:15.451 python[55052:18286619] 2026-06-17 01:33:15.451573 [W:onnxruntime:, graph.cc:4885 CleanUnusedInitializersAndNodeArgs] Removing initializer 'safe_name_14'. It is not used by any node and should be removed from the model.
2026-06-17 01:33:15.451 python[55052:18286619] 2026-06-17 01:33:15.451693 [W:onnxruntime:, graph.cc:4885 CleanUnusedInitializersAndNodeArgs] Removing initializer 'safe_name_13'. It is not used by any node and should be removed from the model.
2026-06-17 01:33:15.475 python[55052:18286619] 2026-06-17 01:33:15.475585 [W:onnxruntime:, graph.cc:4885 CleanUnusedInitializersAndNodeArgs] Removing initializer 'safe_name_16'. It is not used by any node and should be removed from the model.
2026-06-17 01:33:15.475 python[55052:18286619] 2026-06-17 01:33:15.475698 [W:onnxruntime:, graph.cc:4885 CleanUnusedInitializersAndNodeArgs] Removing initializer 'safe_name_15'. It is not used by any node and should be removed from the model.


{'task': 'task003',
 'candidate': '6406.18',
 'file': '/Users/kaiikeda/Programming/Kaggle/Neuro Golf/solution/6406.18/task003.onnx',
 'filesize': 5000,
 'status': 'ok',
 'memory': 1172,
 'params': 18,
 'cost': 1190.0,
 'score': 17.918291413894426,
 'error': ''}

Candidate ONNX files: 29


,candidate_file,status,score,score_delta,cost,cost_delta,memory,memory_delta,params,params_delta,filesize,filesize_delta,error
0,task003_best_v79_baseline.onnx,ok,18.185457,0.267166,911.0,-279.0,888,-284,23,5,7733,2733,
1,task003_rebuild_v100_out_zero_from_ch0_p2_c02....,ok,18.185457,0.267166,911.0,-279.0,888,-284,23,5,5814,814,
2,task003_rebuild_v101_out_zero_from_ch0_p2_c20....,ok,18.185457,0.267166,911.0,-279.0,888,-284,23,5,5814,814,
3,task003_rebuild_v102_out_zero_from_ch0_p2_c22....,ok,18.185457,0.267166,911.0,-279.0,888,-284,23,5,5814,814,
4,task003_rebuild_v80_r0r3only_p2_c02_noconcat_p...,ok,18.185457,0.267166,911.0,-279.0,888,-284,23,5,7733,2733,
5,task003_rebuild_v99_out_zero_from_ch0_p2_c00.onnx,ok,18.185457,0.267166,911.0,-279.0,888,-284,23,5,5648,648,
6,task003_rebuild_v82_r0r3only_p2_c22_noconcat_p...,ok,18.185457,0.267166,911.0,-279.0,888,-284,23,5,7733,2733,
7,task003_rebuild_v81_r0r3only_p2_c20_noconcat_p...,ok,18.185457,0.267166,911.0,-279.0,888,-284,23,5,7733,2733,
8,task003_rebuild_v79_r0r3only_p2_c00_noconcat_p...,ok,18.185457,0.267166,911.0,-279.0,888,-284,23,5,7733,2733,
9,task003_rebuild_v76_r0r3only_p2_c02_noconcat_p...,ok,18.184360,0.266069,912.0,-278.0,889,-283,23,5,7714,2714,


## Candidate Comparison

In [4]:
base_validation = run_model_on_examples(ONNX_PATH, TASK_ID, DATA_DIR)
candidate_validation_rows = []
candidate_validations = {}

for candidate_path in CANDIDATE_ONNX_PATHS:
    report = run_model_on_examples(candidate_path, TASK_ID, DATA_DIR)
    candidate_validations[candidate_path.name] = report
    counts = report.get("counts", {})
    candidate_validation_rows.append({
        "candidate_file": candidate_path.name,
        "validation_status": report.get("status"),
        "pass": counts.get("pass", 0),
        "fail": counts.get("fail", 0),
        "error": counts.get("error", 0),
        "skipped_large_grid": counts.get("skipped_large_grid", 0),
        "first_failure": report.get("first_failure"),
    })

print("Base validation:")
display({k: v for k, v in base_validation.items() if k != "rows"})

print("Candidate validations:")
try:
    import pandas as pd

    validation_df = pd.DataFrame(candidate_validation_rows)
    if candidate_score_rows:
        score_df = pd.DataFrame(candidate_score_rows)
        compare_df = validation_df.merge(
            score_df[["candidate_file", "status", "score", "score_delta", "cost_delta", "memory_delta", "params_delta", "filesize_delta"]],
            on="candidate_file",
            how="left",
        )
        display(compare_df.sort_values(["validation_status", "score_delta"], ascending=[True, False]).reset_index(drop=True))
    else:
        display(validation_df)
except ImportError:
    candidate_validation_rows


2026-06-17 01:33:15.749 python[55052:18286619] 2026-06-17 01:33:15.749044 [W:onnxruntime:, graph.cc:4885 CleanUnusedInitializersAndNodeArgs] Removing initializer 'safe_name_16'. It is not used by any node and should be removed from the model.
2026-06-17 01:33:15.749 python[55052:18286619] 2026-06-17 01:33:15.749139 [W:onnxruntime:, graph.cc:4885 CleanUnusedInitializersAndNodeArgs] Removing initializer 'safe_name_15'. It is not used by any node and should be removed from the model.
2026-06-17 01:33:16.126 python[55052:18286619] 2026-06-17 01:33:16.126334 [W:onnxruntime:, graph.cc:4885 CleanUnusedInitializersAndNodeArgs] Removing initializer 'safe_name_14'. It is not used by any node and should be removed from the model.
2026-06-17 01:33:16.126 python[55052:18286619] 2026-06-17 01:33:16.126720 [W:onnxruntime:, graph.cc:4885 CleanUnusedInitializersAndNodeArgs] Removing initializer 'safe_name_13'. It is not used by any node and should be removed from the model.
2026-06-17 01:33:16.305 pyth

Base validation:


{'status': 'ok', 'counts': {'pass': 265}, 'first_failure': None}

Candidate validations:


,candidate_file,validation_status,pass,fail,error,skipped_large_grid,first_failure,status,score,score_delta,cost_delta,memory_delta,params_delta,filesize_delta
0,task003_best_v79_baseline.onnx,ok,265,0,0,0,None,ok,18.185457,0.267166,-279.0,-284,5,2733
1,task003_rebuild_v100_out_zero_from_ch0_p2_c02....,ok,265,0,0,0,None,ok,18.185457,0.267166,-279.0,-284,5,814
2,task003_rebuild_v101_out_zero_from_ch0_p2_c20....,ok,265,0,0,0,None,ok,18.185457,0.267166,-279.0,-284,5,814
3,task003_rebuild_v102_out_zero_from_ch0_p2_c22....,ok,265,0,0,0,None,ok,18.185457,0.267166,-279.0,-284,5,814
4,task003_rebuild_v79_r0r3only_p2_c00_noconcat_p...,ok,265,0,0,0,None,ok,18.185457,0.267166,-279.0,-284,5,2733
5,task003_rebuild_v80_r0r3only_p2_c02_noconcat_p...,ok,265,0,0,0,None,ok,18.185457,0.267166,-279.0,-284,5,2733
6,task003_rebuild_v81_r0r3only_p2_c20_noconcat_p...,ok,265,0,0,0,None,ok,18.185457,0.267166,-279.0,-284,5,2733
7,task003_rebuild_v82_r0r3only_p2_c22_noconcat_p...,ok,265,0,0,0,None,ok,18.185457,0.267166,-279.0,-284,5,2733
8,task003_rebuild_v99_out_zero_from_ch0_p2_c00.onnx,ok,265,0,0,0,None,ok,18.185457,0.267166,-279.0,-284,5,648
9,task003_rebuild_v75_r0r3only_p2_c00_noconcat_p...,ok,265,0,0,0,None,ok,18.184360,0.266069,-278.0,-283,5,2714


In [5]:
valid_improvements = []
for row in candidate_score_rows:
    validation = candidate_validations.get(row["candidate_file"], {})
    if validation.get("status") == "ok" and row.get("score_delta", 0) > 0:
        valid_improvements.append(row)

print(f"Valid score improvements: {len(valid_improvements)}")
try:
    import pandas as pd

    if valid_improvements:
        display(pd.DataFrame(valid_improvements).sort_values("score_delta", ascending=False).reset_index(drop=True))
except ImportError:
    valid_improvements


Valid score improvements: 29


,task,candidate,file,filesize,status,memory,params,cost,score,error,candidate_file,candidate_path,filesize_delta,memory_delta,params_delta,cost_delta,score_delta,base_status,candidate_status
0,task003_best_v79_baseline,task003,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...,7733,ok,888,23,911.0,18.185457,,task003_best_v79_baseline.onnx,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...,2733,-284,5,-279.0,0.267166,ok,ok
1,task003_rebuild_v100_out_zero_from_ch0_p2_c02,task003,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...,5814,ok,888,23,911.0,18.185457,,task003_rebuild_v100_out_zero_from_ch0_p2_c02....,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...,814,-284,5,-279.0,0.267166,ok,ok
2,task003_rebuild_v101_out_zero_from_ch0_p2_c20,task003,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...,5814,ok,888,23,911.0,18.185457,,task003_rebuild_v101_out_zero_from_ch0_p2_c20....,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...,814,-284,5,-279.0,0.267166,ok,ok
3,task003_rebuild_v102_out_zero_from_ch0_p2_c22,task003,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...,5814,ok,888,23,911.0,18.185457,,task003_rebuild_v102_out_zero_from_ch0_p2_c22....,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...,814,-284,5,-279.0,0.267166,ok,ok
4,task003_rebuild_v80_r0r3only_p2_c02_noconcat_p...,task003,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...,7733,ok,888,23,911.0,18.185457,,task003_rebuild_v80_r0r3only_p2_c02_noconcat_p...,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...,2733,-284,5,-279.0,0.267166,ok,ok
5,task003_rebuild_v99_out_zero_from_ch0_p2_c00,task003,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...,5648,ok,888,23,911.0,18.185457,,task003_rebuild_v99_out_zero_from_ch0_p2_c00.onnx,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...,648,-284,5,-279.0,0.267166,ok,ok
6,task003_rebuild_v82_r0r3only_p2_c22_noconcat_p...,task003,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...,7733,ok,888,23,911.0,18.185457,,task003_rebuild_v82_r0r3only_p2_c22_noconcat_p...,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...,2733,-284,5,-279.0,0.267166,ok,ok
7,task003_rebuild_v81_r0r3only_p2_c20_noconcat_p...,task003,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...,7733,ok,888,23,911.0,18.185457,,task003_rebuild_v81_r0r3only_p2_c20_noconcat_p...,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...,2733,-284,5,-279.0,0.267166,ok,ok
8,task003_rebuild_v79_r0r3only_p2_c00_noconcat_p...,task003,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...,7733,ok,888,23,911.0,18.185457,,task003_rebuild_v79_r0r3only_p2_c00_noconcat_p...,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...,2733,-284,5,-279.0,0.267166,ok,ok
9,task003_rebuild_v76_r0r3only_p2_c02_noconcat_p...,task003,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...,7714,ok,889,23,912.0,18.184360,,task003_rebuild_v76_r0r3only_p2_c02_noconcat_p...,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...,2714,-283,5,-278.0,0.266069,ok,ok


In [6]:
failed_candidates = {
    name: report.get("first_failure")
    for name, report in candidate_validations.items()
    if report.get("first_failure")
}
failed_candidates


{}

## ONNX Structure

In [7]:
model_summary = summarize_model(ONNX_PATH)

print(f"file         : {model_summary['filename']}")
print(f"filesize     : {model_summary['filesize']}")
print(f"ir_version   : {model_summary['ir_version']}")
print(f"opset_imports: {model_summary['opset_imports']}")
print(f"inputs       : {model_summary['inputs']}")
print(f"outputs      : {model_summary['outputs']}")
print(f"nodes        : {model_summary['num_nodes']}")
print(f"initializers : {model_summary['num_initializers']}")
print(f"op_counts    : {model_summary['op_counts']}")

file         : task003.onnx
filesize     : 5000
ir_version   : 8
opset_imports: [{'domain': '', 'version': 10}]
inputs       : [{'name': 'input', 'elem_type': 'FLOAT', 'shape': [1, 10, 30, 30]}]
outputs      : [{'name': 'output', 'elem_type': 'FLOAT16', 'shape': [1, 10, 30, 30]}]
nodes        : 42
initializers : 12
op_counts    : {'Cast': 7, 'Concat': 4, 'Less': 2, 'Mul': 10, 'Not': 2, 'Pad': 1, 'ReduceSum': 2, 'Slice': 5, 'Sub': 5, 'Sum': 3, 'Where': 1}


## Initializers

In [8]:
try:
    import pandas as pd

    display(pd.DataFrame(model_summary["initializers"]))
except ImportError:
    model_summary["initializers"]

,name,dtype,dims,numel,bytes,sample
0,_pt69_one,FLOAT16,[1],1,2,[1.0]
1,_pt69_zero,FLOAT16,[1],1,2,[0.0]
2,_pt69_half,FLOAT16,[1],1,2,[0.5]
3,_pt69_c1s,INT64,[1],1,8,[1]
4,_pt69_c2s,INT64,[1],1,8,[2]
5,_pt69_c3s,INT64,[1],1,8,[3]
6,_pt69_rs0,INT64,[1],1,8,[0]
7,_pt69_re3,INT64,[1],1,8,[4]
8,_boolmask_where_zero,FLOAT16,[],1,2,[0.0]
9,_pt69_sliced_direct_axes_123,INT64,[3],3,24,"[1, 2, 3]"


## Nodes

In [9]:
try:
    import pandas as pd

    nodes_df = pd.DataFrame(model_summary["nodes"])
    display(nodes_df[["index", "op_type", "name", "inputs", "outputs", "attributes"]])
except ImportError:
    model_summary["nodes"][:20]

,index,op_type,name,inputs,outputs,attributes
0,0,Slice,54,"[input, _pt69_ch1_direct_start, _pt69_ch1_dire...",[_pt69_ch1],{}
1,1,Cast,_pt69_ch1__h16,[_pt69_ch1],[_pt69_ch1__h16],{'to': 10}
2,2,Slice,55,"[_pt69_ch1, _pt69_rs0, _pt69_c1s, _pt69_c2s]",[_pt69_r0],{}
3,3,Cast,_pt69_r0__h16,[_pt69_r0],[_pt69_r0__h16],{'to': 10}
4,4,Slice,56,"[_pt69_ch1, _pt69_c1s, _pt69_c2s, _pt69_c2s]",[_pt69_r1],{}
5,5,Cast,_pt69_r1__h16,[_pt69_r1],[_pt69_r1__h16],{'to': 10}
6,6,Slice,57,"[_pt69_ch1, _pt69_c2s, _pt69_c3s, _pt69_c2s]",[_pt69_r2],{}
7,7,Cast,_pt69_r2__h16,[_pt69_r2],[_pt69_r2__h16],{'to': 10}
8,8,Slice,58,"[_pt69_ch1, _pt69_c3s, _pt69_re3, _pt69_c2s]",[_pt69_r3],{}
9,9,Cast,_pt69_r3__h16,[_pt69_r3],[_pt69_r3__h16],{'to': 10}


## GPT Package

In [10]:
model_markdown = compact_model_markdown(model_summary, score_row=score_row, max_nodes=160)
prompt = gpt_analysis_prompt(task_summary, model_markdown)

prompt_path = WORK_DIR / f"{TASK_ID}_gpt_analysis_prompt.md"
prompt_path.write_text(prompt)

print(f"Saved GPT prompt: {prompt_path}")
print(f"Prompt characters: {len(prompt)}")

Saved GPT prompt: /Users/kaiikeda/Programming/Kaggle/Neuro Golf/outputs/gpt_onnx_analysis/task003_gpt_analysis_prompt.md
Prompt characters: 38810


In [11]:
print(prompt)

You are helping optimize a NeuroGolf ONNX model.

Rules:
- Input tensor: float32 [1, 10, 30, 30], name 'input'.
- Output tensor: float32 [1, 10, 30, 30], name 'output'.
- Static tensor shapes only.
- One input and one output only.
- Banned ops: Loop, Scan, NonZero, Unique, Script, Function, Compress.
- Optimize score by reducing memory + params.
- Assume public examples pass unless told otherwise.

Requested output:
1. Summarize what the current ONNX appears to do.
2. Identify redundant or expensive parts.
3. Propose safe rewrite candidates ranked by risk and likely score gain.
4. Do not write code yet.

Task summary:
- task: task003
- train examples: 3
- test examples: 1
- arc-gen examples: 261
- color counts: {0: 5961, 1: 2386, 2: 3578}
- example shapes: [{'subset': 'train', 'index': 0, 'input_shape': '6x3', 'output_shape': '9x3', 'input_cells': 18, 'output_cells': 27}, {'subset': 'train', 'index': 1, 'input_shape': '6x3', 'output_shape': '9x3', 'input_cells': 18, 'output_cells': 27}

Suggested workflow: paste the generated prompt into GPT first and ask for analysis only. After choosing a candidate rewrite, ask GPT to produce a `build_taskXXX.py` script that creates a new ONNX.